<a href="https://colab.research.google.com/github/AlperYildirim1/TS-MechInterp/blob/main/Pareto_Sweep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import platform
import subprocess
import json
import datetime
import time
import torch

try:
    import psutil
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "psutil", "-q"])
    import psutil

OUTPUT_DIR = '/content/drive/MyDrive/MechInterp_All_Results_Final'
os.makedirs(OUTPUT_DIR, exist_ok=True)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
env_save_path = os.path.join(OUTPUT_DIR, f"SAE_Env_Log_Pareto_Sweep_{timestamp}.json")

def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True).strip()
    except Exception as e:
        return f"Command failed: {str(e)}"

judgement_log = {
    "1_TIME_AND_SPACE": {
        "timestamp_local": datetime.datetime.now().isoformat(),
        "timestamp_utc": datetime.datetime.utcnow().isoformat(),
        "timezone": time.tzname,
    },
    "2_OS_AND_KERNEL": {
        "system": platform.system(),
        "release": platform.release(),
        "version": platform.version(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "architecture": platform.architecture(),
        "linux_distribution": run_cmd("cat /etc/os-release | grep PRETTY_NAME").replace('PRETTY_NAME=', '').strip('"')
    },
    "3_HARDWARE_CPU_RAM": {
        "cpu_model_exact": run_cmd("cat /proc/cpuinfo | grep 'model name' | uniq").replace("model name\t: ", ""),
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_threads": psutil.cpu_count(logical=True),
        "ram_total_gb": round(psutil.virtual_memory().total / (1024**3), 2),
        "ram_available_gb": round(psutil.virtual_memory().available / (1024**3), 2),
        "disk_total_gb": round(psutil.disk_usage('/').total / (1024**3), 2),
        "disk_free_gb": round(psutil.disk_usage('/').free / (1024**3), 2),
    },
    "4_HARDWARE_GPU_AND_CUDA": {
        "cuda_is_available": torch.cuda.is_available(),
        "cuda_device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "gpus": {},
        "nvidia_smi_dump": run_cmd("nvidia-smi") if torch.cuda.is_available() else "No GPU found"
    },
    "5_PYTORCH_DEEP_CONFIG": {
        "torch_version": torch.__version__,
        "cuda_compiled_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "nccl_version": torch.cuda.nccl.version() if torch.cuda.is_available() else "N/A",
        "is_bf16_supported": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
        "flash_sdp_enabled": torch.backends.cuda.flash_sdp_enabled(),
        "math_sdp_enabled": torch.backends.cuda.math_sdp_enabled(),
    },
    "6_PYTHON_RUNTIME": {
        "python_executable": sys.executable,
        "python_version": sys.version,
    },
    "7_PIP_FREEZE": run_cmd(f"{sys.executable} -m pip freeze").split('\n'),
    "8_ENVIRONMENT_VARIABLES": dict(os.environ)
}

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        judgement_log["4_HARDWARE_GPU_AND_CUDA"]["gpus"][f"gpu_{i}"] = {
            "name": props.name,
            "vram_total_gb": round(props.total_memory / (1024**3), 2),
            "compute_capability": f"{props.major}.{props.minor}",
            "multi_processor_count": props.multi_processor_count,
            "l2_cache_size_mb": round(props.l2_cache_size / (1024**2), 2) if hasattr(props, 'l2_cache_size') else "Unknown"
        }

with open(env_save_path, 'w') as f:
    json.dump(judgement_log, f, indent=4)

print(f"{'='*60}")
print(f" SAE ENVIRONMENT LOG")
print(f"{'='*60}")
print(f"OS      : {judgement_log['2_OS_AND_KERNEL']['linux_distribution']}")
print(f"CPU     : {judgement_log['3_HARDWARE_CPU_RAM']['cpu_model_exact']} ({judgement_log['3_HARDWARE_CPU_RAM']['logical_threads']} Cores)")
print(f"RAM     : {judgement_log['3_HARDWARE_CPU_RAM']['ram_total_gb']} GB Total")
print(f"PyTorch : {judgement_log['5_PYTORCH_DEEP_CONFIG']['torch_version']} (CUDA {judgement_log['5_PYTORCH_DEEP_CONFIG']['cuda_compiled_version']})")
if torch.cuda.is_available():
    for i in range(judgement_log["4_HARDWARE_GPU_AND_CUDA"]["cuda_device_count"]):
        gpu = judgement_log["4_HARDWARE_GPU_AND_CUDA"]["gpus"][f"gpu_{i}"]
        print(f"GPU [{i}] : {gpu['name']} | {gpu['vram_total_gb']} GB | Compute {gpu['compute_capability']} | bf16: {judgement_log['5_PYTORCH_DEEP_CONFIG']['is_bf16_supported']}")
print(f"\nSaved to: {env_save_path}")
print(f"{'='*60}")

## CONFIG

In [ ]:
!pip install -q x-transformers

import os
import sys
import re
import glob
import json
import traceback
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from x_transformers import Encoder
from tqdm.auto import tqdm
from google.colab import runtime


SEARCH_DIR = '/content/drive/MyDrive/Master_Training/Checkpoints'
OUTPUT_DIR = '/content/drive/MyDrive/MechInterp_All_Results_Final/Pareto_Curves'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LAMBDAS = [0.1, 0.01, 0.001, 0.0001]
FACTORS =[0.5, 4.0]

DATASETS_CONFIG = {
    "weather":     {"path": "weather/weather.csv",             "split": "70_10_20",            "channels": 21,  "batch_size": 128, "d_model": 64},
    "electricity": {"path": "electricity/electricity.csv",     "split": "70_10_20",            "channels": 321, "batch_size": 64,  "d_model": 128},
    "traffic":     {"path": "traffic/traffic.csv",             "split": "70_10_20",            "channels": 862, "batch_size": 16,  "d_model": 96},
    "exchange":    {"path": "exchange_rate/exchange_rate.csv", "split": "70_10_20",            "channels": 8,   "batch_size": 128, "d_model": 16},
    "etth1":       {"path": "ETT-small/ETTh1.csv",             "split":[8640, 2880, 2880],    "channels": 7,   "batch_size": 128, "d_model": 16},
    "etth2":       {"path": "ETT-small/ETTh2.csv",             "split":[8640, 2880, 2880],    "channels": 7,   "batch_size": 128, "d_model": 16},
    "ettm1":       {"path": "ETT-small/ETTm1.csv",             "split":[34560, 11520, 11520], "channels": 7,   "batch_size": 128, "d_model": 16},
    "ettm2":       {"path": "ETT-small/ETTm2.csv",             "split":[34560, 11520, 11520], "channels": 7,   "batch_size": 128, "d_model": 16},
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## ARCHITECTURE

In [ ]:
class TSDataset(Dataset):
    def __init__(self, data, seq_len=336, pred_len=96):
        self.data = data; self.seq_len = seq_len; self.pred_len = pred_len
    def __len__(self): return len(self.data) - self.seq_len - self.pred_len + 1
    def __getitem__(self, idx):
        s = idx + self.seq_len
        return torch.FloatTensor(self.data[idx:s]), torch.FloatTensor(self.data[s:s + self.pred_len])

class RevIN(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.affine_weight = nn.Parameter(torch.ones(num_features))
        self.affine_bias = nn.Parameter(torch.zeros(num_features))
    def forward(self, x, mode):
        if mode == 'norm':
            self.mean = x.mean(dim=1, keepdim=True).detach()
            self.stdev = (x.var(dim=1, keepdim=True, unbiased=False) + self.eps).sqrt().detach()
            x = (x - self.mean) / self.stdev
            return x * self.affine_weight + self.affine_bias
        else:
            x = (x - self.affine_bias) / (self.affine_weight + self.eps)
            return x * self.stdev + self.mean

class AblationPatchTST(nn.Module):
    def __init__(self, seq_len=336, pred_len=96, patch_len=16, stride=8, d_model=128, transformer_depth=1, dropout=0.2, n_channels=7, use_rope=True, use_w_pos=False, use_glu=False):
        super().__init__()
        self.seq_len = seq_len; self.pred_len = pred_len; self.patch_len = patch_len; self.stride = stride; self.d_model = d_model
        self.num_patches = (seq_len - patch_len) // stride + 2
        self.use_w_pos = use_w_pos
        self.revin = RevIN(n_channels)
        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_dropout = nn.Dropout(dropout)
        if self.use_w_pos: self.W_pos = nn.Parameter(torch.randn(1, self.num_patches, d_model) * 0.02)
        transformer_heads = max(1, d_model // 8)
        self.transformer = Encoder(dim=d_model, depth=transformer_depth, heads=transformer_heads, attn_dim_head=16, ff_mult=2, attn_flash=True, rotary_pos_emb=use_rope, use_rmsnorm=True, ff_glu=use_glu, attn_dropout=dropout, ff_dropout=dropout)
        self.flatten_dim = self.num_patches * d_model
        self.head_drop = nn.Dropout(dropout)
        self.head = nn.Linear(self.flatten_dim, pred_len)

    def forward(self, x):
        x = self.revin(x, 'norm')
        B, L, C = x.shape
        x = x.permute(0, 2, 1).reshape(B * C, L, 1)
        x = x.transpose(1, 2)
        x = F.pad(x, (0, self.stride), 'replicate')
        patches = x.unfold(-1, self.patch_len, self.stride).squeeze(1)
        raw_embed = self.patch_embed(patches)
        features = self.pos_dropout(raw_embed + self.W_pos[:, :raw_embed.shape[1], :]) if self.use_w_pos else self.pos_dropout(raw_embed)
        features = self.transformer(features)
        flattened = self.head_drop(features.reshape(features.shape[0], -1))
        out = self.head(flattened).unsqueeze(-1)
        out = out.reshape(B, C, self.pred_len).permute(0, 2, 1)
        return self.revin(out, 'denorm')

class SparseAutoencoder(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.encoder = nn.Linear(d_in, d_hidden)
        self.decoder = nn.Linear(d_hidden, d_in)
        with torch.no_grad():
            self.decoder.weight.data = F.normalize(self.decoder.weight.data, dim=0)
    def forward(self, x):
        f = F.relu(self.encoder(x))
        return self.decoder(f), f


## TRAINING

In [ ]:
# 3. RADAR & AUTO-RESUME
print("📡 Scanning Drive for trained models and existing Pareto plots...")
all_pt_files = glob.glob(os.path.join(SEARCH_DIR, "**/*.pt"), recursive=True)
existing_plots = glob.glob(os.path.join(OUTPUT_DIR, "Pareto_Frontier_*_H*.png"))

completed_runs = set()
for plot_file in existing_plots:
    match = re.search(r'Pareto_Frontier_(.+)_H(\d+)\.png', os.path.basename(plot_file))
    if match:
        completed_runs.add((match.group(1).lower(), int(match.group(2))))

models_to_process =[]
for file_path in all_pt_files:
    path_lower = file_path.lower()
    if "sae_" in path_lower: continue

    detected_ds = next((ds for ds in DATASETS_CONFIG.keys() if ds in path_lower), None)
    h_match = re.search(r'_h(\d+)', path_lower)

    if detected_ds and h_match:
        hz = int(h_match.group(1))
        if (detected_ds, hz) in completed_runs: continue
        models_to_process.append({"path": file_path, "dataset": detected_ds, "horizon": hz})

print(f"🛑 Found {len(completed_runs)} already completed Pareto curves. Skipping them!")
print(f"✅ Found {len(models_to_process)} remaining models to process:")
for m in models_to_process:
    print(f"  - [{m['dataset'].upper()}] Horizon: {m['horizon']}")
print("-" * 50)


# 4. MASTER SWEEP LOOP
try:
    for m_idx, meta in enumerate(models_to_process):
        ds_name = meta['dataset']; horizon = meta['horizon']; model_path = meta['path']
        cfg = DATASETS_CONFIG[ds_name]
        ff_dim = cfg["d_model"] * 2

        print(f"\n🚀[{m_idx+1}/{len(models_to_process)}] STARTING PARETO SWEEP: {ds_name.upper()} | Horizon {horizon}")

        # A. LOAD DATA
        url = "https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/main/" + cfg["path"]
        local_path = f"/content/data_{ds_name}.csv"
        if not os.path.exists(local_path):
            df = pd.read_csv(url)
            df.to_csv(local_path, index=False)
        else:
            df = pd.read_csv(local_path)

        cols_data = df.drop(columns=[df.columns[0]]).values.astype(np.float32)
        n_total = len(cols_data)

        if cfg["split"] == "70_10_20":
            train_end = int(n_total * 0.7)
        else:
            train_end = cfg["split"][0]

        scaler = StandardScaler()
        scaler.fit(cols_data[:train_end])
        all_data = scaler.transform(cols_data)
        train_loader = DataLoader(TSDataset(all_data[:train_end], 336, horizon), batch_size=cfg["batch_size"], shuffle=True, drop_last=True)

        # B. LOAD BASE MODEL
        model = AblationPatchTST(seq_len=336, pred_len=horizon, d_model=cfg["d_model"], n_channels=cfg["channels"]).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        for param in model.parameters(): param.requires_grad = False
        target_module = next((m for m in model.modules() if isinstance(m, nn.GELU)), None)

        # C. HARVEST ACTIVATIONS
        print("   🌾 Harvesting Train Activations...")
        activation_buffer =[]
        total_patches = 0
        MAX_PATCHES = 1_000_000

        def harvest_hook(m, i, o):
            global total_patches
            if total_patches < MAX_PATCHES:
                activation_buffer.append(o.detach().cpu().half())
                total_patches += o.shape[0] * o.shape[1]

        handle = target_module.register_forward_hook(harvest_hook)
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
            for x, _ in train_loader:
                model(x.to(device))
                if total_patches >= MAX_PATCHES: break
        handle.remove()

        all_activations = torch.cat(activation_buffer, dim=0).reshape(-1, ff_dim).float().to(device)
        sae_loader = DataLoader(TensorDataset(all_activations), batch_size=8192, shuffle=True)

        # Free up RAM
        del model, activation_buffer, train_loader, all_data, cols_data, df
        torch.cuda.empty_cache()

        # D. SAE SWEEP
        results = {f: {"L0": [], "MSE":[]} for f in FACTORS}

        for factor in FACTORS:
            d_hidden = int(ff_dim * factor)
            print(f"   ↳ Sweeping {factor}x Dictionary (Size: {d_hidden})")

            for lam in LAMBDAS:
                sae = SparseAutoencoder(ff_dim, d_hidden).to(device)
                optimizer = torch.optim.Adam(sae.parameters(), lr=1e-3)

                best_sae_loss = float('inf')
                patience = 0
                for epoch in range(25): # Shorter max epochs for sweeps
                    epoch_loss = 0
                    for (bx,) in sae_loader:

                        optimizer.zero_grad()
                        x_recon, f = sae(bx)
                        loss = F.mse_loss(x_recon, bx) + (lam * f.abs().sum(dim=-1).mean())
                        loss.backward()
                        optimizer.step()
                        with torch.no_grad(): sae.decoder.weight.data = F.normalize(sae.decoder.weight.data, dim=0)
                        epoch_loss += loss.item()

                    avg_loss = epoch_loss / len(sae_loader)
                    if avg_loss < best_sae_loss - 1e-5:
                        best_sae_loss = avg_loss
                        patience = 0
                    else:
                        patience += 1
                    if patience >= 3: break

                # Evaluate
                sae.eval()
                recon_mse_sum, l0_sum, batches = 0, 0, 0
                with torch.no_grad():
                    for (bx,) in sae_loader:

                        x_recon, f = sae(bx)
                        recon_mse_sum += F.mse_loss(x_recon, bx).item()
                        l0_sum += (f > 1e-5).float().sum(dim=-1).mean().item()
                        batches += 1

                avg_recon = recon_mse_sum / batches
                avg_l0 = l0_sum / batches

                results[factor]["L0"].append(avg_l0)
                results[factor]["MSE"].append(avg_recon)
                print(f"      - Lambda: {lam:.4f} | L0: {avg_l0:5.1f} | MSE: {avg_recon:.4f}")

                del sae, optimizer, x_recon, f, bx
                torch.cuda.empty_cache()

        # E. SAVE RAW DATA (JSON)
        json_path = os.path.join(OUTPUT_DIR, f"Pareto_Data_{ds_name}_H{horizon}.json")
        with open(json_path, 'w') as f:
            json.dump(results, f, indent=4)

        # F. PLOT PARETO CURVE
        plt.style.use('seaborn-v0_8-whitegrid')
        fig, ax = plt.subplots(figsize=(8, 6))

        colors = {0.5: '#1f77b4', 4.0: '#d62728'}
        markers = {0.5: 'o', 4.0: 's'}

        for factor in FACTORS:
            sort_idx = np.argsort(results[factor]["L0"])
            l0_sorted = np.array(results[factor]["L0"])[sort_idx]
            mse_sorted = np.array(results[factor]["MSE"])[sort_idx]
            ax.plot(l0_sorted, mse_sorted, marker=markers[factor], color=colors[factor],
                    linewidth=2, markersize=8, label=f"{factor}x Dictionary", alpha=0.8)

        ax.set_title(f"SAE Pareto Frontier ({ds_name.upper()}, H={horizon})", fontsize=14, pad=15)
        ax.set_xlabel("$L_0$ (Average Active Latents per Token)", fontsize=12)
        ax.set_ylabel("Reconstruction MSE", fontsize=12)
        ax.legend(fontsize=12)
        ax.grid(True, linestyle='--', alpha=0.7)

        plt.tight_layout()
        save_path = os.path.join(OUTPUT_DIR, f"Pareto_Frontier_{ds_name}_H{horizon}.png")
        plt.savefig(save_path, dpi=300)
        plt.close(fig)

        del sae_loader, all_activations
        torch.cuda.empty_cache()
        print(f"✅ Finished {ds_name.upper()} H{horizon}. Saved plot & JSON.")

    print("\n🎉 MASTER PARETO PIPELINE COMPLETE! All results and plots saved to:", OUTPUT_DIR)

except Exception as e:
    print(f"\n❌ FATAL ERROR IN PIPELINE: {e}")
    traceback.print_exc()

finally:
    print("\n🛑 Execution finished. Forcing Google Drive to sync...")
    from google.colab import drive
    drive.flush_and_unmount()
    print("✅ Sync complete. Nuking runtime to save Compute Units...")
    runtime.unassign()